# Exploratory analysis of the MDT data

**Purpose.** This notebook examines the minimization-of-drive-tests (MDT) records before the
calibration. It describes the schema, the missing values, and the distributions. It then examines
the target in groups, in relation to the geometry, in time, and in space. The last section
lists the findings and their consequences for the calibration.

**Inputs.**

| File | Content |
|---|---|
| `configs/config.yaml` | The configuration. `src.utils.load_config` composes it with the other files of `configs/`. |
| `data/raw/cells.csv` | One row for each cell |
| `data/raw/mdt.csv` | One row for each MDT record |
| Sionna RT scene `simulation.scene.name` | Geometry for the building height map |

**Target.** The target of the calibration is the reported RSRP of one (record, cell) pair,
$\widehat{\mathrm{RSRP}}_{r,c}$ in dBm. `src.data.mdt_to_long` gives one row for each
received pair.

**Method.** All quantities come from the input files and from the height map. The notebook
contains no fixed value of the data. It thus remains valid for a run with a different
configuration, and for real MDT data with the same columns.

**Figures.** `configure_matplotlib` sets the figure style. Each figure is saved as a PNG file
in `reports/figures/` with the prefix `eda_`. The text below each figure is its caption.

**Requirements.** Run `task scenario` first. The notebook needs an NVIDIA GPU because it
loads the Sionna RT scene.

## 1. Setup

The first cell imports the packages, reads the configuration of the run, and calls
`setup_notebook`. The second cell defines the plot functions that the sections share.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import pdist

from src.core import MDT_POSITION_COLUMNS, read_cells, rsrp_column
from src.data import mdt_to_long
from src.simulation import building_height_map, load_rt_scene
from src.utils import (
    COLORS,
    DATA_DIR,
    FIGURE_WIDTH_DOUBLE,
    load_config,
    save_figure,
    save_table,
    setup_notebook,
)

RAW_DIR = DATA_DIR / "raw"
cfg = load_config()
device = setup_notebook(cfg)

In [ ]:
def label_panels(axes):
    "Put the labels (a), (b), ... above the top left corner of each panel."
    for index, ax in enumerate(np.ravel(axes)):
        ax.set_title(f"({chr(ord('a') + index)})", loc="left", fontweight="bold")


def format_map(ax):
    "Give a map panel an equal scale on the two axes and axis labels in meters."
    ax.set(xlabel="$x$ (m)", ylabel="$y$ (m)", aspect="equal")
    ax.grid(False)


def ecdf(values):
    "Return the sorted values and the empirical cumulative distribution function."
    ordered = np.sort(np.asarray(values))
    return ordered, np.arange(1, ordered.size + 1) / ordered.size

## 2. Load data

`read_cells` reads `cells.csv`. `mdt.csv` is read with the `timestamp` column as a datetime.
`load_rt_scene` loads the scene, and `building_height_map` gives the height of the highest
surface above the ground on a grid of tiles. A tile is a building if its height is more than
`scenario.grid.min_building_height_m`. Table 1 gives the size of the inputs.

In [ ]:
cells = read_cells(RAW_DIR / "cells.csv")
cells_frame = pd.read_csv(RAW_DIR / "cells.csv")
mdt = pd.read_csv(RAW_DIR / "mdt.csv", parse_dates=["timestamp"])
cell_names = [cell.name for cell in cells]
rsrp_columns = [rsrp_column(name) for name in cell_names]
rsrp = mdt[rsrp_columns].set_axis(cell_names, axis=1)
cell_table = cells_frame.set_index("cell")
site_order = sorted(cells_frame["site"].unique())
site_color = {site: COLORS[index] for index, site in enumerate(site_order)}
cell_color = [site_color[cell.site] for cell in cells]

scene = load_rt_scene(cfg.simulation.scene.name)
height_map = building_height_map(scene, cfg.scenario.grid.tile_size_m)
heights = height_map.heights
ny, nx = heights.shape
size = height_map.tile_size
extent = [
    height_map.x_min,
    height_map.x_min + nx * size,
    height_map.y_min,
    height_map.y_min + ny * size,
]
is_building = heights > cfg.scenario.grid.min_building_height_m


def grid_index(x, y):
    "Return the row and the column of the grid tile that contains each position."
    col = np.clip(((np.asarray(x) - height_map.x_min) // size).astype(int), 0, nx - 1)
    row = np.clip(((np.asarray(y) - height_map.y_min) // size).astype(int), 0, ny - 1)
    return row, col


save_table(
    pd.DataFrame(
        [
            {
                "Input": name,
                "Rows": frame.shape[0],
                "Columns": frame.shape[1],
                "Memory (kB)": frame.memory_usage(deep=True).sum() / 1e3,
            }
            for name, frame in {"cells.csv": cells_frame, "mdt.csv": mdt}.items()
        ]
        + [
            {
                "Input": f"Height map of {cfg.simulation.scene.name}",
                "Rows": ny,
                "Columns": nx,
                "Memory (kB)": heights.nbytes / 1e3,
            }
        ]
    )
    .style.set_caption("Table 1. Size of the inputs.")
    .format({"Memory (kB)": "{:.1f}"})
    .hide(axis="index"),
    "eda_inputs",
)

## 3. Schema and dtypes

Table 2 gives each column of `mdt.csv` with its type, unit, and number of values. The format
of `mdt.csv` is fixed in `src.core.mdt`: the columns `timestamp`, `x`, `y`, `z`, then one
`rsrp_<cell>` column for each row of `cells.csv`. Table 3 gives the columns of `cells.csv`. A
column with one distinct value has the same value for all cells.

In [ ]:
expected_columns = [*MDT_POSITION_COLUMNS, *rsrp_columns]
print(f"The columns of mdt.csv agree with the MDT format: {list(mdt.columns) == expected_columns}")

units = {"timestamp": "-", "x": "m", "y": "m", "z": "m"} | {
    column: "dBm" for column in rsrp_columns
}
meanings = {
    "timestamp": "Start of the time slot",
    "x": "Reported UE position, x axis of the scene",
    "y": "Reported UE position, y axis of the scene",
    "z": "UE position, z axis of the scene",
} | {rsrp_column(name): f"Reported RSRP of {name}" for name in cell_names}
display(
    save_table(
        pd.DataFrame(
            {
                "Type": mdt.dtypes.astype(str),
                "Unit": pd.Series(units),
                "Description": pd.Series(meanings),
                "Values": mdt.notna().sum(),
                "Distinct values": mdt.nunique(),
            }
        ).style.set_caption("Table 2. Columns of mdt.csv."),
        "eda_mdt_columns",
    )
)
save_table(
    pd.DataFrame(
        {
            "Type": cells_frame.dtypes.astype(str),
            "Distinct values": cells_frame.nunique(),
            "First value": cells_frame.iloc[0],
        }
    ).style.set_caption("Table 3. Columns of cells.csv."),
    "eda_cells_columns",
)

## 4. Missingness

The position columns must have no missing value. A missing RSRP has a known cause: the UE
received no path from the cell, or the measured RSRP was below the threshold
$\mathrm{RSRP}_{\min}$ (`scenario.measurement.min_rsrp_dbm`). A missing RSRP is thus left-censored
and missing not at random (MNAR). The probability of a missing value must thus increase with
the distance $d$ from the cell. It must also increase with the horizontal angle $\phi$
between the azimuth of the cell and the direction to the UE. The main lobe of a cell is $|\phi| \le 180^\circ / S$,
where $S$ is the number of cells of a site.

Table 4 gives the missing fraction of each cell and the median distance of the received
and the missing pairs. Fig. 1 shows the pattern and the relation to the distance.

In [ ]:
positions = mdt[["x", "y", "z"]].to_numpy()
cell_xyz = cells_frame[["x", "y", "z"]].to_numpy()
offset = positions[:, None, :] - cell_xyz[None, :, :]
all_distance = np.linalg.norm(offset, axis=2)
bearing = np.degrees(np.arctan2(offset[..., 1], offset[..., 0]))
all_angle = (bearing - cells_frame["azimuth_deg"].to_numpy() + 180) % 360 - 180
received = rsrp.notna().to_numpy()
main_lobe_deg = 180 / len(cfg.scenario.sites.sector_azimuths_deg)
in_main_lobe = np.abs(all_angle) <= main_lobe_deg

position_missing = int(mdt[list(MDT_POSITION_COLUMNS)].isna().sum().sum())
print(f"Missing values in the position columns: {position_missing}")
print(f"Missing fraction of all (record, cell) pairs: {1 - received.mean():.3f}")

missing_table = pd.DataFrame(
    {
        "Missing fraction": 1 - received.mean(axis=0),
        "Median distance, received (m)": [
            np.median(all_distance[received[:, i], i]) for i in range(len(cells))
        ],
        "Median distance, missing (m)": [
            np.median(all_distance[~received[:, i], i]) if (~received[:, i]).any() else np.nan
            for i in range(len(cells))
        ],
    },
    index=pd.Index(cell_names, name="Cell"),
)
save_table(
    missing_table.style.set_caption("Table 4. Missing RSRP of each cell.").format(
        precision=3, na_rep="-"
    ),
    "eda_missing_rsrp",
)

In [ ]:
distance_edges = np.geomspace(all_distance.min(), all_distance.max(), 13)
distance_centers = np.sqrt(distance_edges[:-1] * distance_edges[1:])
distance_bin = np.clip(np.digitize(all_distance, distance_edges) - 1, 0, distance_centers.size - 1)


def detection_by_distance(chosen):
    "Return the fraction of received pairs in each distance bin, for the chosen pairs."
    counts = np.bincount(distance_bin[chosen], minlength=distance_centers.size)
    hits = np.bincount(
        distance_bin[chosen], weights=received[chosen], minlength=distance_centers.size
    )
    return np.where(counts > 0, hits / np.maximum(counts, 1), np.nan)


fig, (ax_bar, ax_matrix, ax_detect) = plt.subplots(
    1, 3, figsize=(FIGURE_WIDTH_DOUBLE, 2.7), width_ratios=[1.1, 1, 1.2]
)
ax_bar.bar(range(len(cells)), missing_table["Missing fraction"], color=cell_color)
ax_bar.set_xticks(range(len(cells)), cell_names, rotation=90)
ax_bar.set(ylabel="Missing fraction", ylim=(0, 1))
ax_bar.grid(axis="x", visible=False)

order = np.argsort(mdt["timestamp"].to_numpy(), kind="stable")
ax_matrix.imshow(received[order], aspect="auto", cmap="Greys", interpolation="nearest")
ax_matrix.set_xticks(range(len(cells)), cell_names, rotation=90)
ax_matrix.set(ylabel="Record, in the order of the time")
ax_matrix.grid(False)

ax_detect.plot(distance_centers, detection_by_distance(in_main_lobe), marker="o", label="Main lobe")
ax_detect.plot(
    distance_centers,
    detection_by_distance(~in_main_lobe),
    marker="s",
    linestyle="--",
    label="Outside the main lobe",
)
ax_detect.set(
    xscale="log", xlabel="Distance to the cell $d$ (m)", ylabel="Detection probability", ylim=(0, 1)
)
ax_detect.legend()
label_panels([ax_bar, ax_matrix, ax_detect])
save_figure(fig, "eda_missingness")
plt.show()

near_detection = detection_by_distance(np.ones_like(received))
detection_rho = stats.spearmanr(all_distance.ravel(), received.ravel()).statistic
print(f"Spearman correlation between the distance and the detection: {detection_rho:+.3f}")

**Fig. 1.** Missing RSRP. (a) Missing fraction of each cell. The color identifies the
site. (b) Received (black) and missing (white) pairs. Each row is one record, in the order of
the time. (c) Fraction of the pairs with a reported RSRP in logarithmic distance bins, in the
main lobe and outside the main lobe of the cell.

## 5. Univariate distributions

Table 5 gives the statistics of each variable. The UE height above the ground is
$z - z_{\mathrm{ground}}$, where $z_{\mathrm{ground}}$ comes from the height map. The skewness
and the excess kurtosis are zero for a normal distribution. Fig. 2 shows the distributions.
The bins of the RSRP histogram have the width of the RSRP resolution
`scenario.measurement.rsrp_resolution_db`.

In [ ]:
ue_height = mdt["z"] - height_map.z_ground
received_rsrp = rsrp.to_numpy()[received]
step = cfg.scenario.measurement.rsrp_resolution_db
variables = {
    "x (m)": mdt["x"].to_numpy(),
    "y (m)": mdt["y"].to_numpy(),
    "UE height above ground (m)": ue_height.to_numpy(),
    "RSRP, all received pairs (dBm)": received_rsrp,
}
save_table(
    pd.DataFrame(
        {
            name: {
                "Count": values.size,
                "Mean": values.mean(),
                "Std": values.std(ddof=1),
                "Min": values.min(),
                "Median": np.median(values),
                "Max": values.max(),
                "Skewness": stats.skew(values),
                "Excess kurtosis": stats.kurtosis(values),
            }
            for name, values in variables.items()
        }
    )
    .T.style.set_caption("Table 5. Univariate statistics.")
    .format(precision=2)
    .format("{:.0f}", subset=["Count"]),
    "eda_univariate",
)

In [ ]:
rsrp_edges = np.arange(received_rsrp.min() - step / 2, received_rsrp.max() + step, step)
fig, axes = plt.subplots(1, 4, figsize=(FIGURE_WIDTH_DOUBLE, 2.2))
axes[0].hist(mdt["x"], bins=30, color=COLORS[0])
axes[0].set(xlabel="$x$ (m)", ylabel="Number of records")
axes[1].hist(mdt["y"], bins=30, color=COLORS[0])
axes[1].set(xlabel="$y$ (m)")
axes[2].hist(ue_height, bins=30, color=COLORS[2])
axes[2].set(xlabel="UE height above ground (m)")
axes[3].hist(received_rsrp, bins=rsrp_edges, color=COLORS[5])
axes[3].axvline(
    cfg.scenario.measurement.min_rsrp_dbm, color="gray", linestyle=":", label="Threshold"
)
axes[3].set(xlabel="RSRP (dBm)", ylabel="Number of pairs")
axes[3].legend()
label_panels(axes)
save_figure(fig, "eda_univariate")
plt.show()

**Fig. 2.** Univariate distributions. (a) Reported $x$. (b) Reported $y$. (c) UE height
above the ground. (d) Reported RSRP of all received pairs. The dotted line is
`scenario.measurement.min_rsrp_dbm`.

## 6. Target analysis

The target is the reported RSRP of a (record, cell) pair. The generator gives

$$\widehat{\mathrm{RSRP}}_{r,c} = \Delta \cdot \mathrm{round}\!\left(
\frac{\mathrm{RSRP}_{r,c} + w_{r,c}}{\Delta} \right), \qquad
w_{r,c} \sim \mathcal{N}(\mu_w, \sigma_w^2),$$

and removes the pair if $\widehat{\mathrm{RSRP}}_{r,c} < \mathrm{RSRP}_{\min}$. Three
properties of the target follow. It is discrete with the step $\Delta$. It is truncated on the
left at $\mathrm{RSRP}_{\min}$. It has a bias $\mu_w$ that a calibration must not attribute to
the propagation. Table 6 measures these properties. Fig. 3 compares the target with a normal
distribution and shows the margin between the strongest and the second strongest cell of a
record.

In [ ]:
long = mdt_to_long(mdt, cells)
target = long["rsrp_dbm"].to_numpy()
ranked = np.sort(np.where(received, rsrp.to_numpy(), -np.inf), axis=1)[:, ::-1]
strongest = ranked[:, 0]
has_second = np.isfinite(ranked[:, 1])
margin = strongest[has_second] - ranked[has_second, 1]
threshold = cfg.scenario.measurement.min_rsrp_dbm
noise_std = cfg.scenario.measurement.rsrp_noise_std_db
noise_mean = cfg.scenario.measurement.rsrp_noise_mean_db

save_table(
    pd.Series(
        {
            "Received (record, cell) pairs": len(long),
            "Records": long["record"].nunique(),
            "Mean number of cells in a record": received.sum(axis=1).mean(),
            "Distinct RSRP values": np.unique(target).size,
            "Smallest step between distinct values (dB)": np.diff(np.unique(target)).min(),
            "Configured resolution (dB)": step,
            "Fraction of pairs at the threshold": np.mean(target == threshold),
            "Fraction of pairs less than one noise std above the threshold": np.mean(
                target < threshold + noise_std
            ),
            "Configured mean of the measurement error (dB)": noise_mean,
            "Median RSRP of the strongest cell (dBm)": np.median(strongest),
            "Fraction of records with a second cell": has_second.mean(),
            "Median margin to the second cell (dB)": np.median(margin) if margin.size else np.nan,
        }
    )
    .round(3)
    .to_frame("Value")
    .style.set_caption("Table 6. Properties of the target."),
    "eda_target",
)

In [ ]:
fig, (ax_cdf, ax_qq, ax_margin) = plt.subplots(1, 3, figsize=(FIGURE_WIDTH_DOUBLE, 2.5))
ax_cdf.plot(*ecdf(target), label="All received pairs")
ax_cdf.plot(*ecdf(strongest), linestyle="--", label="Strongest cell")
ax_cdf.axvline(threshold, color="gray", linestyle=":", label="Threshold")
ax_cdf.set(xlabel="RSRP (dBm)", ylabel="Empirical CDF", ylim=(0, 1))
ax_cdf.legend(loc="upper left")

(quantiles, ordered), (slope, intercept, _) = stats.probplot(target, dist="norm")
ax_qq.scatter(quantiles, ordered, s=3, color=COLORS[0], linewidths=0)
ax_qq.plot(quantiles, intercept + slope * quantiles, color=COLORS[1], label="Normal fit")
ax_qq.axhline(threshold, color="gray", linestyle=":")
ax_qq.set(xlabel="Normal quantile", ylabel="RSRP (dBm)")
ax_qq.legend(loc="upper left")

if margin.size:
    ax_margin.hist(margin, bins=np.arange(-step / 2, margin.max() + step, step), color=COLORS[5])
ax_margin.set(xlabel="Strongest minus second cell (dB)", ylabel="Number of records")
label_panels([ax_cdf, ax_qq, ax_margin])
save_figure(fig, "eda_target")
plt.show()

**Fig. 3.** Target. (a) Empirical CDF of the reported RSRP of all received pairs and of
the strongest cell of each record. The dotted line is the threshold. (b) Normal Q-Q plot of
the reported RSRP. A deviation in the lower tail shows the truncation at the threshold.
(c) Difference between the RSRP of the strongest and of the second strongest cell, for the
records with two or more cells.

## 7. Group structure

The data has three group levels: the cell, the site of the cell, and the azimuth of the cell.
The share of the target variance that a grouping explains is

$$\eta^2 = \frac{\sum_g n_g (\bar{y}_g - \bar{y})^2}{\sum_i (y_i - \bar{y})^2},$$

where $n_g$ and $\bar{y}_g$ are the size and the mean of group $g$. Table 7 gives $\eta^2$
for each grouping. Table 8 gives the statistics of each cell. Fig. 4 shows the target in each
group. The pairs of one record are not independent, so $\eta^2$ is a descriptive measure, not
a test.

In [ ]:
long["site"] = cell_table.loc[long["cell"], "site"].to_numpy()
long["azimuth_deg"] = cell_table.loc[long["cell"], "azimuth_deg"].to_numpy()
pair_cell_xyz = cell_table.loc[long["cell"], ["x", "y", "z"]].to_numpy()
pair_delta = long[["x", "y", "z"]].to_numpy() - pair_cell_xyz
long["distance_m"] = np.linalg.norm(pair_delta, axis=1)
pair_bearing = np.degrees(np.arctan2(pair_delta[:, 1], pair_delta[:, 0]))
long["angle_deg"] = (pair_bearing - long["azimuth_deg"] + 180) % 360 - 180


def eta_squared(values, groups):
    "Return the share of the variance of the values that the group means explain."
    grand = values.mean()
    summary = values.groupby(groups).agg(["count", "mean"])
    between = (summary["count"] * (summary["mean"] - grand) ** 2).sum()
    return between / ((values - grand) ** 2).sum()


display(
    save_table(
        pd.DataFrame(
            {
                "Groups": {key: long[key].nunique() for key in ["cell", "site", "azimuth_deg"]},
                "eta^2": {
                    key: eta_squared(long["rsrp_dbm"], long[key])
                    for key in ["cell", "site", "azimuth_deg"]
                },
            }
        )
        .rename(index={"cell": "Cell", "site": "Site", "azimuth_deg": "Azimuth"})
        .style.set_caption("Table 7. Share of the target variance that each grouping explains.")
        .format({"eta^2": "{:.3f}"}),
        "eda_variance_share",
    )
)
per_cell = (
    long.groupby("cell")
    .agg(
        Pairs=("rsrp_dbm", "size"),
        Mean=("rsrp_dbm", "mean"),
        Std=("rsrp_dbm", "std"),
        Median=("rsrp_dbm", "median"),
        Min=("rsrp_dbm", "min"),
        Max=("rsrp_dbm", "max"),
        **{"Median distance (m)": ("distance_m", "median")},
    )
    .reindex(cell_names)
)
save_table(
    per_cell.style.set_caption("Table 8. Reported RSRP of each cell (dBm).")
    .format(precision=1, na_rep="-")
    .format("{:.0f}", subset=["Pairs"], na_rep="-"),
    "eda_rsrp_by_cell",
)

In [ ]:
fig, (ax_cell, ax_azimuth) = plt.subplots(
    1, 2, figsize=(FIGURE_WIDTH_DOUBLE, 2.6), width_ratios=[2, 1]
)
box_style = {
    "patch_artist": True,
    "medianprops": {"color": "black"},
    "flierprops": {"markersize": 2},
}
cell_values = [long.loc[long["cell"] == name, "rsrp_dbm"].to_numpy() for name in cell_names]
boxes = ax_cell.boxplot(cell_values, tick_labels=cell_names, **box_style)
for patch, color in zip(boxes["boxes"], cell_color, strict=True):
    patch.set_facecolor(color)
ax_cell.tick_params(axis="x", rotation=90)
ax_cell.set(ylabel="RSRP (dBm)")
ax_cell.grid(axis="x", visible=False)

azimuths = sorted(long["azimuth_deg"].unique())
azimuth_values = [long.loc[long["azimuth_deg"] == a, "rsrp_dbm"].to_numpy() for a in azimuths]
boxes = ax_azimuth.boxplot(azimuth_values, tick_labels=[f"{a:g}" for a in azimuths], **box_style)
for patch in boxes["boxes"]:
    patch.set_facecolor(COLORS[5])
ax_azimuth.set(xlabel="Azimuth of the cell (\N{DEGREE SIGN})", ylabel="RSRP (dBm)")
ax_azimuth.grid(axis="x", visible=False)
label_panels([ax_cell, ax_azimuth])
save_figure(fig, "eda_groups")
plt.show()

**Fig. 4.** Target in groups. (a) Reported RSRP of each cell. The color identifies the
site. (b) Reported RSRP of the cells with the same azimuth, for all sites. The boxes show the
quartiles, and the whiskers extend to 1.5 times the interquartile range.

## 8. Relationships

**Distance.** A log-distance model gives the mean RSRP of cell $c$ at the distance $d$,

$$\mathrm{RSRP}_c(d) = A_c - 10\, n_c \log_{10} d,$$

where $A_c$ is an offset and $n_c$ is the path-loss exponent. A least-squares line of the
reported RSRP against $10 \log_{10} d$ gives $A_c$ and the apparent exponent $n_c$ for each
cell. The fit uses only the received pairs. The truncation at the threshold removes the weak
pairs at large $d$, so the apparent $n_c$ is smaller than the exponent of the full data. The
residual $e_{r,c}$ is the reported RSRP minus the fit. Its standard deviation is the
shadowing plus the measurement error. Table 9 gives the fit of each cell.

**Angle.** The residual against the horizontal angle $\phi$ shows the effect of the antenna
pattern that the distance model does not include.

**Line of sight.** The test uses the straight line from the cell to the UE. The pair is
in line of sight (LoS) on the height map if no tile on this line is higher than the line. The samples within one tile
of the UE are not used, because the GNSS error can move a reported position into a building
tile. Assumption: this test ignores the shape of the roofs and the terrain. It is an
approximation of the LoS of the ray tracer. Table 10 compares the LoS and the non-line-of-sight
(NLoS) pairs.

**Cells.** The Pearson correlation between the RSRP of two cells uses the records that
contain both cells. Fig. 5 shows the three relationships.

In [ ]:
# A line fit with a residual needs a minimum of three points.
MIN_FIT_PAIRS = 3
# Assumption: a correlation from fewer common records is too noisy to show.
MIN_COMMON_RECORDS = 10

log_distance = 10 * np.log10(long["distance_m"])
fits = {}
for name, chosen in long.groupby("cell").groups.items():
    if len(chosen) < MIN_FIT_PAIRS:
        continue
    slope, intercept = np.polyfit(log_distance[chosen], long.loc[chosen, "rsrp_dbm"], 1)
    residual = long.loc[chosen, "rsrp_dbm"] - (intercept + slope * log_distance[chosen])
    fits[name] = {
        "Pairs": len(chosen),
        "Offset A (dBm)": intercept,
        "Apparent exponent n": -slope,
        "Residual std (dB)": residual.std(ddof=2),
        "R^2": 1 - residual.var() / long.loc[chosen, "rsrp_dbm"].var(),
    }
    long.loc[chosen, "residual_db"] = residual
fit_table = pd.DataFrame(fits).T.reindex(cell_names)
save_table(
    fit_table.style.set_caption("Table 9. Log-distance fit of each cell.")
    .format(precision=2, na_rep="-")
    .format("{:.0f}", subset=["Pairs"], na_rep="-"),
    "eda_log_distance_fit",
)

In [ ]:
# Spacing of the samples on the line: half a tile, so each tile on the line gets a sample.
SAMPLES_PER_TILE = 2

horizontal = np.hypot(pair_delta[:, 0], pair_delta[:, 1])
fraction = np.linspace(0, 1, int(np.ceil(horizontal.max() / size * SAMPLES_PER_TILE)) + 1)
line_x = pair_cell_xyz[:, [0]] + fraction * pair_delta[:, [0]]
line_y = pair_cell_xyz[:, [1]] + fraction * pair_delta[:, [1]]
line_height = pair_cell_xyz[:, [2]] + fraction * pair_delta[:, [2]] - height_map.z_ground
surface = heights[grid_index(line_x, line_y)]
near_ue = (1 - fraction) * horizontal[:, None] < size
long["los"] = ~((surface > line_height) & ~near_ue).any(axis=1)

los_table = (
    long.groupby("los")
    .agg(
        Pairs=("rsrp_dbm", "size"),
        **{
            "Mean RSRP (dBm)": ("rsrp_dbm", "mean"),
            "Mean residual (dB)": ("residual_db", "mean"),
            "Median distance (m)": ("distance_m", "median"),
        },
    )
    .rename(index={True: "LoS", False: "NLoS"})
)
los_table.insert(1, "Fraction", los_table["Pairs"] / los_table["Pairs"].sum())
display(
    save_table(
        los_table.style.set_caption("Table 10. Pairs in LoS and in NLoS on the height map.")
        .format(precision=2)
        .format("{:.0f}", subset=["Pairs"]),
        "eda_los_nlos",
    )
)
height_rho = stats.spearmanr(
    long["z"] - height_map.z_ground, long["residual_db"], nan_policy="omit"
)
print(f"Spearman correlation between the UE height and the residual: {height_rho.statistic:+.3f}")

In [ ]:
fig, (ax_distance, ax_angle, ax_corr) = plt.subplots(
    1, 3, figsize=(FIGURE_WIDTH_DOUBLE, 2.6), width_ratios=[1.1, 1.1, 1]
)
for value, label, color in [(True, "LoS", COLORS[0]), (False, "NLoS", COLORS[1])]:
    chosen = long["los"] == value
    ax_distance.scatter(
        long.loc[chosen, "distance_m"],
        long.loc[chosen, "rsrp_dbm"],
        s=3,
        color=color,
        linewidths=0,
        alpha=0.6,
        label=label,
    )
ax_distance.axhline(threshold, color="gray", linestyle=":")
ax_distance.set(xscale="log", xlabel="Distance to the cell $d$ (m)", ylabel="RSRP (dBm)")
ax_distance.legend(markerscale=3)

angle_edges = np.linspace(-180, 180, 25)
angle_centers = (angle_edges[:-1] + angle_edges[1:]) / 2
valid = long["residual_db"].notna()
binned = stats.binned_statistic(
    long.loc[valid, "angle_deg"], long.loc[valid, "residual_db"], statistic="mean", bins=angle_edges
)
ax_angle.scatter(
    long.loc[valid, "angle_deg"], long.loc[valid, "residual_db"], s=2, color=COLORS[5], linewidths=0
)
ax_angle.plot(angle_centers, binned.statistic, color=COLORS[1], label="Mean in a bin")
for edge in (-main_lobe_deg, main_lobe_deg):
    ax_angle.axvline(edge, color="gray", linestyle="--")
ax_angle.set(xlabel="Angle to the azimuth $\\phi$ (\N{DEGREE SIGN})", ylabel="Residual $e$ (dB)")
ax_angle.legend()

correlation = rsrp.corr(min_periods=MIN_COMMON_RECORDS)
image = ax_corr.imshow(correlation, vmin=-1, vmax=1, cmap="RdBu_r")
ax_corr.set_xticks(range(len(cells)), cell_names, rotation=90)
ax_corr.set_yticks(range(len(cells)), cell_names)
ax_corr.grid(False)
fig.colorbar(image, ax=ax_corr, label="Pearson correlation", shrink=0.8)
label_panels([ax_distance, ax_angle, ax_corr])
save_figure(fig, "eda_relationships")
plt.show()

main_residual = long.loc[valid & (long["angle_deg"].abs() <= main_lobe_deg), "residual_db"].mean()
side_residual = long.loc[valid & (long["angle_deg"].abs() > main_lobe_deg), "residual_db"].mean()
print(f"Mean residual in the main lobe: {main_residual:+.2f} dB. Outside: {side_residual:+.2f} dB.")

**Fig. 5.** Relationships. (a) Reported RSRP against the distance to the cell, for the
pairs in LoS and in NLoS on the height map. The dotted line is the threshold. (b) Residual of
the log-distance fit against the horizontal angle $\phi$ to the azimuth of the cell. The
dashed lines are the limits of the main lobe. (c) Correlation between the RSRP of two cells.
A white square has too few common records.

## 9. Temporal and spatial structure

**Time.** The scene and the cells do not change in time. The number of records and the mix of
positions change during the day (Section 4.2 of `00_scenario.ipynb`). The residual of the
distance fit must thus have no relation to the hour. A Kruskal-Wallis test compares the
residuals of the hours. Fig. 6 shows the records, the strongest RSRP, and the residual for each
hour of the day.

**Space.** Shadowing has a spatial correlation. For the residuals of one cell, the empirical
semivariogram at the lag $h$ is

$$\hat{\gamma}(h) = \frac{1}{2 |N(h)|} \sum_{(i,j) \in N(h)} (e_i - e_j)^2,$$

where $N(h)$ is the set of pairs of records with a horizontal separation in the lag bin of
$h$. The pairs of all cells go into one estimate. Gudmundson (Electronics Letters, 1991, doi:10.1049/el:19911328) gives an
exponential correlation model, $\gamma(h) = \sigma^2 \left(1 - e^{-h/d_{\mathrm{c}}}\right)$.
With this model, $\hat{\gamma}$ reaches $(1 - e^{-1})\,\sigma^2$ at the decorrelation
distance $d_{\mathrm{c}}$. The GNSS error and
the measurement error add a constant (nugget) to $\hat{\gamma}$. The largest lag is half of
`scenario.sites.isd_m`. Fig. 7 shows the mean RSRP of the strongest cell on a grid and the
semivariogram. Table 11 gives the summary.

In [ ]:
hour = mdt["timestamp"].dt.hour.to_numpy()
long["hour"] = hour[long["record"].to_numpy()]
hours = np.arange(24)
records_per_hour = np.bincount(hour, minlength=hours.size)
strongest_by_hour = pd.Series(strongest).groupby(hour).median().reindex(hours)
residual_by_hour = long.groupby("hour")["residual_db"].agg(["mean", "sem"]).reindex(hours)
hour_groups = [group.dropna().to_numpy() for _, group in long.groupby("hour")["residual_db"]]
hour_groups = [group for group in hour_groups if group.size > 0]
kruskal = stats.kruskal(*hour_groups) if len(hour_groups) > 1 else None

fig, (ax_count, ax_strong, ax_res) = plt.subplots(1, 3, figsize=(FIGURE_WIDTH_DOUBLE, 2.3))
ax_count.bar(hours, records_per_hour, color=COLORS[5])
ax_count.set(xlabel="Hour of day", ylabel="Number of records")
ax_strong.plot(hours, strongest_by_hour, marker="o")
ax_strong.set(xlabel="Hour of day", ylabel="Median strongest RSRP (dBm)")
ax_res.errorbar(
    hours, residual_by_hour["mean"], yerr=2 * residual_by_hour["sem"], marker="o", capsize=2
)
ax_res.axhline(0, color="gray", linestyle=":")
ax_res.set(xlabel="Hour of day", ylabel="Mean residual $e$ (dB)")
for ax in (ax_count, ax_strong, ax_res):
    ax.set_xticks(hours[::6])
label_panels([ax_count, ax_strong, ax_res])
save_figure(fig, "eda_temporal")
plt.show()
if kruskal is not None:
    print(
        "Kruskal-Wallis test of the residual by hour: "
        f"H = {kruskal.statistic:.2f}, p = {kruskal.pvalue:.3f}"
    )

**Fig. 6.** Temporal structure. (a) Number of records in each hour of the day. (b) Median
RSRP of the strongest cell. (c) Mean residual of the log-distance fit with an interval of two
standard errors.

In [ ]:
# Lag bins of the semivariogram. The number of bins is a choice of the display.
NUM_LAG_BINS = 20
# Bin size of the RSRP map: one tenth of the ISD. A choice of the display.
map_bin_m = cfg.scenario.sites.isd_m / 10

lag_edges = np.linspace(0, cfg.scenario.sites.isd_m / 2, NUM_LAG_BINS + 1)
lag_centers = (lag_edges[:-1] + lag_edges[1:]) / 2
gamma_sum = np.zeros(NUM_LAG_BINS)
gamma_count = np.zeros(NUM_LAG_BINS)
fitted = long.dropna(subset=["residual_db"])
for _, group in fitted.groupby("cell"):
    if len(group) < 2:
        continue
    separation = pdist(group[["x", "y"]].to_numpy())
    half_square = 0.5 * pdist(group[["residual_db"]].to_numpy(), "sqeuclidean")
    lag = np.digitize(separation, lag_edges) - 1
    inside = (lag >= 0) & (lag < NUM_LAG_BINS)
    gamma_sum += np.bincount(lag[inside], weights=half_square[inside], minlength=NUM_LAG_BINS)
    gamma_count += np.bincount(lag[inside], minlength=NUM_LAG_BINS)
gamma = np.where(gamma_count > 0, gamma_sum / np.maximum(gamma_count, 1), np.nan)
sill = fitted["residual_db"].var()
reached = np.flatnonzero(gamma >= (1 - np.exp(-1)) * sill)
decorrelation_m = lag_centers[reached[0]] if reached.size else np.nan

x_edges = np.arange(extent[0], extent[1] + map_bin_m, map_bin_m)
y_edges = np.arange(extent[2], extent[3] + map_bin_m, map_bin_m)
strongest_map = stats.binned_statistic_2d(
    mdt["x"], mdt["y"], strongest, statistic="mean", bins=[x_edges, y_edges]
).statistic.T
row, col = grid_index(mdt["x"], mdt["y"])

fig, (ax_map, ax_gamma) = plt.subplots(
    1, 2, figsize=(FIGURE_WIDTH_DOUBLE, 3.0), width_ratios=[1.3, 1]
)
ax_map.imshow(is_building, origin="lower", extent=extent, cmap="Greys", alpha=0.3)
image = ax_map.imshow(
    strongest_map,
    origin="lower",
    extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
    alpha=0.85,
)
ax_map.scatter(
    cells_frame["x"], cells_frame["y"], marker="^", color="black", edgecolor="white", s=45
)
fig.colorbar(image, ax=ax_map, label="Mean strongest RSRP (dBm)", shrink=0.8)
format_map(ax_map)

ax_gamma.plot(lag_centers, gamma, marker="o", label=r"$\hat{\gamma}(h)$")
ax_gamma.axhline(sill, color="gray", linestyle="--", label=r"Residual variance $\sigma^2$")
ax_gamma.axhline(
    (1 - np.exp(-1)) * sill, color="gray", linestyle=":", label=r"$(1 - e^{-1})\,\sigma^2$"
)
ax_gamma.set(xlabel="Separation $h$ (m)", ylabel=r"Semivariance (dB$^2$)", ylim=(0, None))
ax_gamma.legend()
label_panels([ax_map, ax_gamma])
save_figure(fig, "eda_spatial")
plt.show()

save_table(
    pd.Series(
        {
            "Map bin size (m)": map_bin_m,
            "Fraction of reported positions in a building tile": is_building[row, col].mean(),
            "GNSS standard deviation (m)": cfg.scenario.measurement.gnss_std_m,
            "Residual variance (dB^2)": sill,
            "Semivariance in the first lag bin (dB^2)": gamma[0],
            "Decorrelation distance d_c (m)": decorrelation_m,
        }
    )
    .round(3)
    .to_frame("Value")
    .style.set_caption("Table 11. Spatial structure."),
    "eda_spatial",
)

**Fig. 7.** Spatial structure. (a) Mean RSRP of the strongest cell in each map bin. The
gray areas are buildings, and the triangles are the sites. (b) Empirical semivariogram of the
residuals of the log-distance fit. The dotted line is the level that defines the
decorrelation distance.

## 10. Findings

Table 12 lists the findings. The evidence column comes from the sections above, so the table
is correct for each run. Some properties of the data come from the rules of the generator and
do not need a measurement:

1. A missing RSRP is censored at `scenario.measurement.min_rsrp_dbm` or has no path. A calibration loss
   must not ignore the missing pairs and must not impute them as random values.
2. The target is quantized with the step `scenario.measurement.rsrp_resolution_db`. The quantization
   adds a uniform error with the variance $\Delta^2 / 12$.
3. The measurement error has the mean `scenario.measurement.rsrp_noise_mean_db`. A calibration with a
   free offset can absorb it. A calibration of the materials only can attribute it to the
   propagation by mistake.
4. The reported position has a GNSS error with the standard deviation
   `scenario.measurement.gnss_std_m`. The simulated RSRP at the reported position is thus not the
   RSRP at the true position.
5. The records are statistically independent: the data has no UE trajectories.

In [ ]:
finite_exponents = fit_table["Apparent exponent n"].dropna()
los_fraction = los_table["Fraction"].get("LoS", 0.0)
los_gap = los_table["Mean residual (dB)"].get("LoS", np.nan) - los_table["Mean residual (dB)"].get(
    "NLoS", np.nan
)
findings = [
    (
        "Missing RSRP",
        f"{1 - received.mean():.1%} of the pairs are missing. Spearman rho between distance "
        f"and detection: {detection_rho:+.2f}.",
        "Use a censored likelihood or a detection model for the missing pairs.",
    ),
    (
        "Quantization",
        f"Smallest step between distinct values: {np.diff(np.unique(target)).min():g} dB. "
        f"Configured resolution: {step:g} dB.",
        "Compare the simulation with the reported RSRP after the same rounding, or add the "
        "quantization variance.",
    ),
    (
        "Truncation",
        f"{np.mean(target < threshold + noise_std):.1%} of the pairs are less than one noise "
        "std above the threshold.",
        "A loss on the received pairs only is biased upward near the threshold.",
    ),
    (
        "Distance law",
        f"Apparent exponent n from {finite_exponents.min():.2f} to {finite_exponents.max():.2f} "
        f"(median {finite_exponents.median():.2f}). Residual std from "
        f"{fit_table['Residual std (dB)'].min():.2f} to "
        f"{fit_table['Residual std (dB)'].max():.2f} dB.",
        "The residual std is the scale of the error that a calibration can decrease.",
    ),
    (
        "Cell groups",
        f"eta^2: cell {eta_squared(long['rsrp_dbm'], long['cell']):.3f}, "
        f"site {eta_squared(long['rsrp_dbm'], long['site']):.3f}.",
        "Compare the errors of the calibration for each cell, not only in total.",
    ),
    (
        "Line of sight",
        f"{los_fraction:.1%} of the pairs are LoS on the height map. Mean residual LoS minus "
        f"NLoS: {los_gap:+.2f} dB.",
        "Report the calibration error for LoS and NLoS pairs separately.",
    ),
    (
        "Antenna pattern",
        f"Mean residual in the main lobe {main_residual:+.2f} dB, outside {side_residual:+.2f} dB.",
        "A distance-only model does not include the pattern. The ray tracer must include it.",
    ),
    (
        "Time",
        "Kruskal-Wallis p = "
        + (f"{kruskal.pvalue:.3f}" if kruskal is not None else "not available")
        + " for the residual by hour.",
        "A small p shows that the position mix of the hours changes the residual.",
    ),
    (
        "Space",
        f"Decorrelation distance d_c = {decorrelation_m:.0f} m. Semivariance in the first lag "
        f"bin: {gamma[0] / sill:.0%} of the residual variance.",
        "Split the training and the test data in spatial blocks larger than d_c.",
    ),
    (
        "Position error",
        f"{is_building[row, col].mean():.1%} of the reported positions are in a building tile.",
        "Do not remove these records. The true position is outdoor.",
    ),
]
save_table(
    pd.DataFrame(findings, columns=["Finding", "Evidence", "Consequence for the calibration"])
    .style.set_caption("Table 12. Findings.")
    .hide(axis="index")
    .set_properties(**{"text-align": "left"}),
    "eda_findings",
)